# 05 · Confirmation on the locked test split (NF-UNSW-NB15-v3, NF-ToN-IoT-v3, NF-CSE-CIC-IDS2018-v3)

**This notebook uses the locked_test role for the first and only time.** Its design and pass rules were fixed on 1 Oct 2026, before any locked-test result. It must not be rerun with a changed design.

**The method under test: a shaping-monotone flow detector (v2c).**
- The direction in which padding, dummy packets and delay can move each NetFlow field is taken from relations measured on three datasets (notebook 02), not from intuition.
- The five packet-size bins, which these operations move both ways, are replaced by five cumulative counts (packets at least as large as each bin edge), which they can only raise.
- The detector is trained monotone in those directions and keeps the fields the attacker can reach, instead of dropping them.
- Result: undoing any in-budget operation can never raise the score, so every benign verdict is certified by construction against padding up to 100 bytes per packet, up to 5 dummy packets and up to 1,000 ms of delay. The per-flow verifier confirms each one.

**Closest published work, used as context:**
- Incer et al. 2018 (malware) introduced monotone classifiers against evasion and combining them with an ordinary classifier.
- Chen et al. 2021 verified monotonicity for cryptojacking and spam classifiers.
- Li 2026 guarantees intrusion detection by using only fields the attacker cannot reach.

**Detectors compared on the locked test:**
- v1: the notebook 02 detector.
- v2c: shaping-monotone; models from notebook 04, trained here for NF-CSE-CIC-IDS2018-v3 with the same recipe.
- invariant-only: trained only on the fields no declared operation can change. This is the guarantee-by-exclusion approach, also certified by construction.
- two-layer: alert if v1 or v2c flags the flow, each at a 0.5% false-alarm rate on development. Clean detection comes from both layers; the guarantee covers what v2c catches. The design follows Incer et al.'s combined classifier and is evaluated here, not claimed as new.

All thresholds are margin thresholds set on development benign flows, and every verifier is given exactly the verdict threshold, so a verdict and its certificate always compare the same numbers. The false-alarm rate each threshold actually gives on development is recorded (a detector with few distinct scores can land below 1%).

**Attacks on locked-test attack flows each detector flags (up to 2,000 per detector and dataset):**
- random: 40 in-budget tries.
- adaptive: 300 queries, refining the best operation plans.
- Both use the same simulation of padding, dummy packets and delay and score candidates with the detector's exact margin. An evasion of v2c or of the invariant-only detector would contradict their construction; the notebook stops if one is found.
- Flows that themselves break an audited relation (for example a longest packet above 1514 bytes, about 0.01% of rows) are outside the operation model, as for the certificate. They are not attacked, and their count is reported.

**Pass rules, fixed before any locked-test result (distinct records):**
1. v2c's recall at its 1% false-alarm threshold exceeds the invariant-only detector's, with the lower end of a 95% cluster bootstrap interval above zero, on at least 2 of 3 datasets.
2. The two-layer detector's recall is at least v1's recall minus 0.01 on at least 2 of 3 datasets.

**Reported, not scored:**
- Multiplicity-weighted metrics, per-family flag rates and the false-alarm rates the development thresholds give on the locked test.
- Certificates on 5,000 random clean benign locked-test flows per detector.
- Recall under each attack.
- v2b on NF-ToN-IoT-v3 development with a 5-second limit for the flows that hit the 1-second limit in notebook 04.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import gc, hashlib, importlib, json, os, sys, time
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/XIDS_Certify_Research')
REPO = PROJECT / 'xids-certify'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import scipy
import xgboost as xgb
import src.nf3_data as nf
import src.operations as ops
import src.certify as cf
import src.detector as det
import src.attacks as atk
for m in (nf, ops, cf, det, atk):
    importlib.reload(m)

TABLES = REPO / 'results' / 'tables'
MODELS5 = PROJECT / 'models' / 'nf05'
CACHE = PROJECT / 'cache' / 'nf05'
CACHE.mkdir(parents=True, exist_ok=True)
DATASETS = ['unsw', 'ton', 'cic2018']
DETECTORS = ['v1', 'invariant', 'v2c', 'two_layer']
N_CERT, N_ATTACK, N_BOOT = 5000, 2000, 1000
TIME_LIMIT = 1.0
SEED = 20261001
FEATS = nf.MODEL_FEATURES
BUDGET = ops.BUDGETS['primary']
INVARIANT = list(ops.UNCHANGED)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 120)

V1 = json.loads((TABLES / 'nf02_models.json').read_text())
V4 = json.loads((TABLES / 'nf04_summary.json').read_text())
MACHINE = cf.machine()
VERSIONS = {'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__, 'xgboost': xgb.__version__}
print('machine:', MACHINE)
print('versions:', VERSIONS)
print('invariant-only inputs:', INVARIANT)

## Detectors and thresholds (development only)

In [ ]:
def frame(df, cols):
    out = df.copy()
    if any(c in cols for c in ops.CUM):
        ops.add_cumulative(out)
    return out[cols].astype(np.float32)

def load(path, sha=None):
    if sha is not None:
        assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == sha, f'{path}: model file changed'
    b = xgb.Booster()
    b.load_model(str(path))
    return b

def margins(ens, X, chunk=500_000):
    return np.concatenate([ens.margin(X[i:i + chunk]) for i in range(0, len(X), chunk)])

def at_fpr(m, y, fpr):
    return det.threshold_at_fpr(m[y == 0], fpr)

def sigmoid(m):
    return 1.0 / (1.0 + np.exp(-m))

def make_verifier(sp, thr, budget=None):
    # a verifier whose threshold is exactly the verdict threshold (no round trip through a probability), so a
    # verdict and its certificate always compare the same numbers
    V = cf.Verifier(sp['ens'], float(sigmoid(thr)), sp['integer'], budget or BUDGET, context=sp['ctx'])
    V.m_thr = float(thr)
    return V

DET, DATA, RECORDS = {}, {}, {}
cols = FEATS + ['row_idx', 'y', 'family', 'multiplicity', 'cluster_id', 'role']
for ds in DATASETS:
    df = nf.load_split(PROJECT, ds, roles=['detector_train', 'development', 'locked_test'], columns=cols)
    tr = df[df['role'] == 'detector_train']
    dv = df[df['role'] == 'development'].reset_index(drop=True)
    lk = df[df['role'] == 'locked_test'].reset_index(drop=True)
    del df
    gc.collect()
    specs = {}
    specs['v1'] = {'feats': V1[ds]['features'], 'booster': load(PROJECT / V1[ds]['model_path'], V1[ds]['model_sha256'])}
    key = f'{ds}/v2c_shaping_monotone'
    v2c_feats = det.variant_features(FEATS, 'v2c_shaping_monotone')
    if key in V4['models']:
        rec = V4['models'][key]
        specs['v2c'] = {'feats': rec['features'], 'booster': load(PROJECT / rec['model_path'], rec['model_sha256'])}
    else:
        b = det.fit(frame(tr[FEATS], v2c_feats), tr['y'].to_numpy(), frame(dv[FEATS], v2c_feats), dv['y'].to_numpy(),
                    monotone_constraints=det.monotone(v2c_feats, 'v2c_shaping_monotone'))
        sha = det.save(b, MODELS5 / f'{ds}_v2c_shaping_monotone_xgb.json')
        specs['v2c'] = {'feats': v2c_feats, 'booster': b, 'trained_here': True, 'sha256': sha}
    b = det.fit(frame(tr[FEATS], INVARIANT), tr['y'].to_numpy(), frame(dv[FEATS], INVARIANT), dv['y'].to_numpy())
    sha = det.save(b, MODELS5 / f'{ds}_invariant_xgb.json')
    specs['invariant'] = {'feats': INVARIANT, 'booster': b, 'trained_here': True, 'sha256': sha}
    for name, sp in specs.items():
        sp['ens'] = cf.Ensemble(sp['booster'], sp['feats'])
        sp['ctx'] = sp['feats'] + [f for f in cf.REQUIRED if f not in sp['feats']] + \
            ([] if (all(c in sp['feats'] for c in ops.CUM) or all(h in sp['feats'] for h in ops.HIST)) else ops.CUM)
        ctx_tr = frame(tr[FEATS].iloc[:400_000], sp['ctx'])
        sp['integer'] = ops.integer_features(ctx_tr, sp['ctx'])
        sp['m_dev'] = margins(sp['ens'], frame(dv[FEATS], sp['feats']).to_numpy())
        sp['m_lk'] = margins(sp['ens'], frame(lk[FEATS], sp['feats']).to_numpy())
        if name == 'v2c':
            mono = det.monotone(sp['feats'], 'v2c_shaping_monotone')
            sp['monotone_violations'] = det.monotone_violations(sp['ens'], frame(tr[FEATS].iloc[:400_000], sp['feats']).to_numpy(),
                                                                sp['feats'], mono)
            assert sp['monotone_violations'] == 0, f'{ds}: v2c breaks its monotone constraints'
        del ctx_tr
    y_dv = dv['y'].to_numpy()
    for name, sp in specs.items():
        for fpr in (0.01, 0.005):
            sp[f'thr_{fpr}'] = float(at_fpr(sp['m_dev'], y_dv, fpr))     # margin threshold used for every verdict
            sp[f'dev_far_{fpr}'] = float((sp['m_dev'][y_dv == 0] >= sp[f'thr_{fpr}']).mean())
        RECORDS[f'{ds}/{name}'] = {'features': sp['feats'], 'trees': len(sp['ens'].trees),
                                   'threshold_margin_1pct': sp['thr_0.01'], 'threshold_margin_0.5pct': sp['thr_0.005'],
                                   'development_false_alarm_1pct': sp['dev_far_0.01'],
                                   'development_false_alarm_0.5pct': sp['dev_far_0.005'],
                                   'distinct_development_margins': int(len(np.unique(sp['m_dev']))),
                                   'model_sha256': sp.get('sha256', (V1[ds] if name == 'v1' else V4['models'].get(f'{ds}/v2c_shaping_monotone', {})).get('model_sha256')),
                                   'trained_here': bool(sp.get('trained_here', False)),
                                   'monotone_violation_share': sp.get('monotone_violations')}
    DET[ds] = specs
    DATA[ds] = {'y_dv': y_dv, 'lk': lk, 'y_lk': lk['y'].to_numpy(), 'dv_ton': dv if ds == 'ton' else None,
                'tr_ton_ctx_v2b': frame(tr[FEATS].iloc[:400_000], det.context_features(FEATS, 'v2b_certifiable')) if ds == 'ton' else None}
    del tr, dv
    gc.collect()
    hy_dev = (specs['v1']['m_dev'] >= specs['v1']['thr_0.005']) | (specs['v2c']['m_dev'] >= specs['v2c']['thr_0.005'])
    print(f"{ds}: detectors ready | development false-alarm rate at the 1% rule: "
          + ", ".join(f"{n} {specs[n]['dev_far_0.01']:.4f}" for n in ('v1', 'invariant', 'v2c'))
          + f" | two-layer {hy_dev[y_dv == 0].mean():.4f} | v2c trained here: {bool(specs['v2c'].get('trained_here', False))}")

## Locked test: detection (distinct records primary, multiplicity reported)

In [ ]:
def verdicts(ds, name):
    sp = DET[ds]
    if name == 'two_layer':
        return (sp['v1']['m_lk'] >= sp['v1']['thr_0.005']) | (sp['v2c']['m_lk'] >= sp['v2c']['thr_0.005'])
    return sp[name]['m_lk'] >= sp[name]['thr_0.01']

def cluster_ci(ds, a, b, rng, B=N_BOOT):
    lk, y = DATA[ds]['lk'], DATA[ds]['y_lk']
    att = y == 1
    cid = pd.factorize(lk['cluster_id'].to_numpy()[att])[0]
    n = np.bincount(cid)
    da = np.bincount(cid, weights=a[att].astype(float))
    db = np.bincount(cid, weights=b[att].astype(float))
    C = len(n)
    diffs = np.empty(B)
    for i in range(B):
        w = np.bincount(rng.integers(0, C, C), minlength=C)
        diffs[i] = ((da - db) * w).sum() / (n * w).sum()
    point = (da - db).sum() / n.sum()
    return point, np.quantile(diffs, 0.025), np.quantile(diffs, 0.975)

from sklearn.metrics import average_precision_score, roc_auc_score
DETECT, FAMS = [], []
rng = np.random.default_rng([SEED, 1])          # bootstrap
for ds in DATASETS:
    lk, y = DATA[ds]['lk'], DATA[ds]['y_lk']
    w = lk['multiplicity'].to_numpy().astype(float)
    for name in DETECTORS:
        pred = verdicts(ds, name)
        for weighting, ww in (('distinct records', np.ones(len(y))), ('multiplicity', w)):
            row = {'dataset': ds, 'detector': name, 'weighting': weighting,
                   'recall': float(ww[(y == 1) & pred].sum() / ww[y == 1].sum()),
                   'false_alarm_rate': float(ww[(y == 0) & pred].sum() / ww[y == 0].sum())}
            if name != 'two_layer':
                s = DET[ds][name]['m_lk']
                row['auroc'] = float(roc_auc_score(y, s, sample_weight=ww))
                row['pr_auc'] = float(average_precision_score(y, s, sample_weight=ww))
            DETECT.append(row)
        f = pd.DataFrame({'family': lk['family'], 'pred': pred}).groupby('family')['pred'].mean().rename(name)
        FAMS.append(f.to_frame().assign(dataset=ds).reset_index())
detect = pd.DataFrame(DETECT)
display(detect[detect['weighting'] == 'distinct records'])
display(detect[detect['weighting'] == 'multiplicity'])
fam = pd.concat(FAMS).groupby(['dataset', 'family']).first().reset_index()
display(fam)

RULES = []
for ds in DATASETS:
    p1, lo1, hi1 = cluster_ci(ds, verdicts(ds, 'v2c'), verdicts(ds, 'invariant'), rng)
    p2, lo2, hi2 = cluster_ci(ds, verdicts(ds, 'two_layer'), verdicts(ds, 'v1'), rng)
    RULES.append({'dataset': ds, 'v2c_minus_invariant_recall': p1, 'ci_low_1': lo1, 'ci_high_1': hi1,
                  'rule1_pass': bool(lo1 > 0), 'two_layer_minus_v1_recall': p2, 'ci_low_2': lo2, 'ci_high_2': hi2,
                  'rule2_pass': bool(p2 >= -0.01)})
rules = pd.DataFrame(RULES)
display(rules)

## Certificates on 5,000 random clean benign locked-test flows per detector

In [ ]:
CERT = []
rng = np.random.default_rng([SEED, 2])          # certificate samples
for ds in DATASETS:
    lk, y = DATA[ds]['lk'], DATA[ds]['y_lk']
    for name in ('v2c', 'invariant', 'two_layer'):
        sp = DET[ds]['v2c' if name == 'two_layer' else name]
        benign = ~verdicts(ds, name) & (y == 0)
        pos = np.sort(rng.permutation(np.flatnonzero(benign))[:N_CERT])
        V = make_verifier(sp, sp['thr_0.005'] if name == 'two_layer' else sp['thr_0.01'])
        X = frame(lk.iloc[pos][FEATS], sp['ctx']).to_numpy()
        res = cf.run(V, X, lk['row_idx'].to_numpy()[pos], CACHE / f'{ds}_{name}_certificates.parquet', TIME_LIMIT)
        row = cf.summarise(res, {'dataset': ds, 'detector': name})
        CERT.append(row)
        print(f"{ds}/{name}: certified {row['certified_share']:.4f} | witness {row['witness']} | unresolved {row['unresolved']}")
cert = pd.DataFrame(CERT)
display(cert)

## Attacks: random (40 tries) and adaptive (300 queries) in-budget shaping of flagged attack flows

In [ ]:
ATT = []
rng = np.random.default_rng([SEED, 3])          # attack samples

def flow_dict(base, i):
    return {k: float(v) for k, v in base.iloc[i].items()}

def in_model(base, pos):
    # flows that themselves break an audited relation are outside the operation model: not attacked, counted
    keep = [i for i in pos if not cf.out_of_model(ops.add_cumulative(flow_dict(base, i)))]
    return np.array(keep, dtype=np.int64), len(pos) - len(keep)

def run_attacks(base, pos, score, seed):
    r2 = np.random.default_rng(seed)
    rand = np.array([atk.random_attack(flow_dict(base, i), BUDGET, score, r2)[0] for i in pos])
    adap = np.array([atk.adaptive_attack(flow_dict(base, i), BUDGET, score, r2)[0] for i in pos])
    return rand, adap

def rate(v):
    return float((v < 0).mean()) if len(v) else 0.0

for ds in DATASETS:
    lk, y = DATA[ds]['lk'], DATA[ds]['y_lk']
    sp = DET[ds]
    base = lk[FEATS]
    for name in DETECTORS:
        t0 = time.time()
        if name == 'two_layer':
            # attack flows the v2c layer flags cannot be evaded (proof), so recall under attack is that floor plus
            # whatever survives among the flows only v1 flags; a sample of floor flows is attacked to confirm it
            score = atk.scorer([sp['v1']['ens'], sp['v2c']['ens']], [sp['v1']['thr_0.005'], sp['v2c']['thr_0.005']],
                               [sp['v1']['feats'], sp['v2c']['feats']])
            floor_mask = sp['v2c']['m_lk'] >= sp['v2c']['thr_0.005']
            v1only = (sp['v1']['m_lk'] >= sp['v1']['thr_0.005']) & ~floor_mask
            floor_recall = float(floor_mask[y == 1].mean())
            v1only_recall = float(v1only[y == 1].mean())
            pos, skipped = in_model(base, np.sort(rng.permutation(np.flatnonzero(v1only & (y == 1)))[:N_ATTACK]))
            rand, adap = run_attacks(base, pos, score, SEED + 7)
            fpos, fskipped = in_model(base, np.sort(rng.permutation(np.flatnonzero(floor_mask & (y == 1)))[:N_ATTACK // 4]))
            frand, fadap = run_attacks(base, fpos, score, SEED + 8)
            assert (frand >= 0).all() and (fadap >= 0).all(), f'{ds}: a floor flow was evaded, contradicting the construction'
            er, ea = rate(rand), rate(adap)
            row = {'dataset': ds, 'detector': name, 'flagged_attacked': len(pos), 'out_of_model_skipped': skipped + fskipped,
                   'floor_flows_attacked': len(fpos),
                   'clean_recall': float(verdicts(ds, name)[y == 1].mean()), 'guaranteed_floor_recall': floor_recall,
                   'random_evasion_rate': er, 'adaptive_evasion_rate': ea,
                   'recall_under_random': floor_recall + v1only_recall * (1 - er),
                   'recall_under_adaptive': floor_recall + v1only_recall * (1 - ea)}
        else:
            score = atk.scorer([sp[name]['ens']], [sp[name]['thr_0.01']], [sp[name]['feats']])
            pos, skipped = in_model(base, np.sort(rng.permutation(np.flatnonzero(verdicts(ds, name) & (y == 1)))[:N_ATTACK]))
            rand, adap = run_attacks(base, pos, score, SEED + 7)
            clean_recall = float(verdicts(ds, name)[y == 1].mean())
            er, ea = rate(rand), rate(adap)
            row = {'dataset': ds, 'detector': name, 'flagged_attacked': len(pos), 'out_of_model_skipped': skipped,
                   'clean_recall': clean_recall,
                   'guaranteed_floor_recall': clean_recall if name in ('v2c', 'invariant') else 0.0,
                   'random_evasion_rate': er, 'adaptive_evasion_rate': ea,
                   'recall_under_random': clean_recall * (1 - er), 'recall_under_adaptive': clean_recall * (1 - ea)}
            if name in ('v2c', 'invariant'):
                assert er == 0 and ea == 0, f'{ds}/{name}: an in-budget evasion contradicts the construction'
        row['seconds'] = time.time() - t0
        ATT.append(row)
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
attacks = pd.DataFrame(ATT)
display(attacks)

## Reported: v2b on NF-ToN-IoT-v3 development with a 5-second limit

In [ ]:
prev = pd.read_parquet(PROJECT / 'cache' / 'nf04' / 'ton_v2b_certifiable_clean_primary.parquet')
slow = prev[(prev['status'] == 'UNRESOLVED') & (prev['stage'] == 'time_limit')]['row_idx'].to_numpy()
rec = V4['models']['ton/v2b_certifiable']
b2 = load(PROJECT / rec['model_path'], rec['model_sha256'])
ens2 = cf.Ensemble(b2, rec['features'])
ctx2 = det.context_features(FEATS, 'v2b_certifiable')
dv = DATA['ton']['dv_ton']
sel = dv[dv['row_idx'].isin(slow)]
V5 = cf.Verifier(ens2, rec['threshold'], ops.integer_features(DATA['ton']['tr_ton_ctx_v2b'], ctx2), BUDGET, context=ctx2)
res5 = (cf.run(V5, frame(sel[FEATS], ctx2).to_numpy(), sel['row_idx'].to_numpy(), CACHE / 'ton_v2b_dev_5s.parquet', 5.0)
        if len(sel) else pd.DataFrame({'status': pd.Series(dtype=str)}))
n_prev = len(prev)
cert_prev = int((prev['status'] == 'CERTIFIED').sum())
v2b5 = {'flows_unresolved_at_1s': len(slow), **{f'at_5s_{k.lower()}': int((res5['status'] == k).sum()) for k in cf.STATUS},
        'certified_share_1s': cert_prev / n_prev,
        'certified_share_if_5s': (cert_prev + int((res5['status'] == 'CERTIFIED').sum())) / n_prev}
print(json.dumps(v2b5, indent=2))

## Decision

In [ ]:
r1 = int(rules['rule1_pass'].sum())
r2 = int(rules['rule2_pass'].sum())
decision = {'rule1_v2c_beats_invariant': {'datasets_passing': r1, 'passes': r1 >= 2},
            'rule2_two_layer_keeps_v1_recall': {'datasets_passing': r2, 'passes': r2 >= 2}}
display(rules[['dataset', 'v2c_minus_invariant_recall', 'ci_low_1', 'ci_high_1', 'rule1_pass',
               'two_layer_minus_v1_recall', 'rule2_pass']])
print(json.dumps(decision, indent=2))
detect.to_csv(TABLES / 'nf05_locked_detection.csv', index=False)
fam.to_csv(TABLES / 'nf05_locked_family_flag_rates.csv', index=False)
rules.to_csv(TABLES / 'nf05_rules.csv', index=False)
cert.to_csv(TABLES / 'nf05_certificates.csv', index=False)
attacks.to_csv(TABLES / 'nf05_attacks.csv', index=False)
(TABLES / 'nf05_summary.json').write_text(json.dumps({
    'decision': decision, 'v2b_ton_5s': v2b5, 'models': RECORDS, 'machine': MACHINE, 'versions': VERSIONS,
    'seed': SEED, 'budget': BUDGET, 'time_limit_seconds': TIME_LIMIT,
    'locked_test_used_once': '2026-10-01'}, indent=2, default=float) + '\n')

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/XIDS_Certify_Research/xids-certify")
msg = "Notebook 05: confirmation on the locked test split (first and only use) for NF-UNSW-NB15-v3, NF-ToN-IoT-v3 and NF-CSE-CIC-IDS2018-v3; shaping-monotone detector v2c against v1, an invariant-only detector and the two-layer detector; certificates, random and adaptive in-budget attacks, cluster bootstrap intervals and decision on the two pre-registered rules; v2b 5-second sensitivity reported"
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)